# PaperMind v4 — Drive parquet → Supabase **Faz 3** warehouse mirror

**Migrationlar**: `db/migrations/0005_paper_estra_temporal.sql` + `0006_paper_metadata.sql` + `0007_method_centrality.sql` + `0008_neighbor_bibcoupling.sql` + (dinamik 0009 `dim_author`)

**Hedef tablolar (15 — küçükten büyüğe sırayla yüklenir):**
1. `fact_method_field_affinity` (390 × 6, A2)
2. `fact_method_topic_affinity` (65,061 × 6, W-19)
3. `dim_paper_replication` (24.87M × 3, B42-019 broad regex)
4. `fact_paper_field` (24.87M × 4, W-30)
5. `fact_paper_interdisc` (24.87M × 5, W-18 Rao-Stirling)
6. `fact_paper_velocity` (24.87M × 5, W-29)
7. `fact_paper_quality_v3` (24.87M × 6, W-31)
8. `fact_paper_w_estra` (24.87M × 15, W-33, ~1 GB)
9. `fact_paper_topic` (~75M rank≤3 loader filter, W-03)
10. `fact_paper_metod` (51.79M × 6, W-05)
11. `fact_paper_disruption` (24.87M × 8, W-20 CD₅)
12. `fact_paper_beauty` (24.87M × 10, W-21)
13. `fact_paper_centrality` (~24.87M corpus subset / 100M parquet anti-join, N09)
14. `dim_author` (22.65M × 22, N15) — schema dinamik 0009 üretimi
15. `fact_paper_bibcoupling_top50` (643M × 5, N09b, NO FK, ~3 saat)

**Tier**: Pro+ + Compute **4XL** (geçici upload süresi). Bitince Small'a düş.
**Beklenen toplam süre**: ~7-8 saat 4XL'de.
**Disk önkoşul**: 80 GB autoscale (Faz 3 sonrası tahmin %75 doluluk).
**Önkoşul**: Faz 1 (PaperCard) + Faz 2 (sentence_role + d_estra + ref_age) PASS.

**B-009 pattern**: VALID_PAPER_IDS anti-join FK guard. paper_id PaperCard'da yoksa skip.

**Restart-safe**: Her tablo `ON CONFLICT DO NOTHING` + per-table COMMIT; oturum koparsa kaldığı yerden devam.

---

## Çalıştırma sırası
1. Cell 1 — Setup (Drive + DB + VALID_PAPER_IDS).
2. Cell 2 — Schema audit (15 parquet) — **ÇIKTIYI OKUYUN, kolon adları row_builder'larla eşleşiyor mu?**
3. **Notebook DIŞI**: `psql $DB_URL < db/migrations/0005_paper_estra_temporal.sql` ... 0006/0007/0008 (lokalden uygula).
4. Cell 3 — 0005-0008 apply doğrulama + 0009 dim_author dinamik üretim+apply.
5. Cell 4 — stream_upload helper.
6. Cell 5..19 — 15 upload (sırayla, en küçük → en büyük).
7. Cell 20 — Verify (row count + KK gate).

## Cell 1 — Setup (Drive mount + install + DB + VALID_PAPER_IDS)

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

!pip -q install psycopg2-binary==2.9.9 pandas==2.2.2 pyarrow==17.0.0

import os, json, math, time, sys
import pandas as pd
import pyarrow.parquet as pq
import psycopg2
from psycopg2.extras import execute_values
from datetime import datetime, timedelta

DB_URL = userdata.get('SUPABASE_DB_URL')
assert DB_URL and DB_URL.startswith('postgresql://'), 'Colab Secrets → SUPABASE_DB_URL (Session Pooler)'

DRIVE_ROOT = '/content/drive/MyDrive/Dataleak'
PATHS = {
    'fact_method_field_affinity':     f'{DRIVE_ROOT}/N09c/fact_method_field_affinity.parquet',
    'fact_method_topic_affinity':     f'{DRIVE_ROOT}/N09c/fact_method_topic_affinity.parquet',
    'dim_paper_replication':          f'{DRIVE_ROOT}/facts/dim_paper_replication.parquet',
    'fact_paper_field':               f'{DRIVE_ROOT}/facts/fact_paper_field.parquet',
    'fact_paper_interdisc':           f'{DRIVE_ROOT}/N09c/fact_paper_interdisc.parquet',
    'fact_paper_velocity':            f'{DRIVE_ROOT}/facts/fact_paper_velocity.parquet',
    'fact_paper_quality_v3':          f'{DRIVE_ROOT}/facts/fact_paper_quality_v3.parquet',
    'fact_paper_w_estra':             f'{DRIVE_ROOT}/facts/fact_paper_w_estra.parquet',
    'fact_paper_topic':               f'{DRIVE_ROOT}/facts/fact_paper_topic.parquet',
    'fact_paper_metod':               f'{DRIVE_ROOT}/facts/fact_paper_metod.parquet',
    'fact_paper_disruption':          f'{DRIVE_ROOT}/N09e/cd_5.parquet',  # ENVANTER §176: dosya adı cd_5.parquet
    'fact_paper_beauty':              f'{DRIVE_ROOT}/N09d/fact_paper_beauty.parquet',
    'fact_paper_centrality':          f'{DRIVE_ROOT}/facts/fact_paper_centrality.parquet',
    'dim_author':                     f'{DRIVE_ROOT}/facts/dim_author.parquet',  # ENVANTER §130: facts/ subdir
    'fact_paper_bibcoupling_top50':   f'{DRIVE_ROOT}/N09b/fact_paper_bibcoupling_top50.parquet',
}

CHUNK = 50_000
PAGE_SIZE = 5_000
BATCH_ROWS = 500_000

print('=== Parquet existence + size ===')
missing = []
for name, path in PATHS.items():
    exists = os.path.exists(path)
    size_gb = os.path.getsize(path) / 1e9 if exists else 0
    flag = '✓' if exists else '✗'
    print(f'  {flag}  {name:32s} {path}  ({size_gb:.2f} GB)')
    if not exists: missing.append(name)
if missing:
    print(f'\n⚠ MISSING: {missing} — Drive yolu doğrula veya N09c/N09b/dims subdir kontrol et.')

print('\n=== Önkoşul + VALID_PAPER_IDS ===')
with psycopg2.connect(DB_URL) as conn, conn.cursor() as cur:
    cur.execute('SELECT version FROM public.schema_migrations ORDER BY applied_at;')
    applied = [r[0] for r in cur.fetchall()]
    print(f'Applied migrations: {applied}')
    for req in ('0001','0002','0003','0004'):
        ok = any(v.startswith(req) for v in applied)
        print(f"  {'✓' if ok else '✗'} migration {req} prereq")
    cur.execute('SELECT COUNT(*) FROM public.fact_paper_id_card;')
    pc_count = cur.fetchone()[0]
    assert pc_count > 24_000_000, f'Faz 1 önkoşul FAIL — PaperCard {pc_count:,} < 24M'
    print(f'fact_paper_id_card: {pc_count:,}')
    cur.execute('SELECT COUNT(*) FROM public.fact_paper_sentence_role;')
    sr = cur.fetchone()[0]
    assert sr > 24_000_000, f'Faz 2 önkoşul FAIL — sentence_role {sr:,} < 24M'
    print(f'fact_paper_sentence_role: {sr:,}')

    print('\nLoading VALID_PAPER_IDS set (anti-join FK filter)...')
    cur.execute('SELECT paper_id FROM public.fact_paper_id_card;')
    VALID_PAPER_IDS = {r[0] for r in cur.fetchall()}
    print(f'  ✓ {len(VALID_PAPER_IDS):,} valid paper_ids loaded into RAM')

# === Restart-safe state + JSONL log (B-009 enrichment) ===
import json as _json
STATE_DIR  = f'{DRIVE_ROOT}/_phase3_state'
os.makedirs(STATE_DIR, exist_ok=True)
STATE_PATH = f'{STATE_DIR}/phase3_state.json'
LOG_PATH   = f'{STATE_DIR}/phase3_log.jsonl'

def _state_load():
    if not os.path.exists(STATE_PATH):
        return {'started_at': datetime.now().isoformat(), 'tables': {}}
    with open(STATE_PATH) as f:
        return _json.load(f)

def _state_save(state):
    tmp = STATE_PATH + '.tmp'
    with open(tmp, 'w') as f:
        _json.dump(state, f, indent=2, default=str)
    os.replace(tmp, STATE_PATH)

def _log(entry):
    entry = {'ts': datetime.now().isoformat(), **entry}
    with open(LOG_PATH, 'a') as f:
        f.write(_json.dumps(entry, default=str) + '\n')

STATE = _state_load()
_log({'phase': 'session_start', 'tables_completed': [k for k,v in STATE['tables'].items() if v.get('status')=='completed']})
print(f"\n=== State checkpoint ({STATE_PATH}) ===")
print(f"  Session started: {STATE['started_at']}")
done = [k for k,v in STATE['tables'].items() if v.get('status')=='completed']
print(f"  Completed tables ({len(done)}): {done}")
in_prog = [k for k,v in STATE['tables'].items() if v.get('status')=='in_progress']
print(f"  In-progress (will resume): {in_prog}")
print(f"  Log: {LOG_PATH}")


## Cell 2 — Schema audit (15 parquet)

**Bu hücreyi mutlaka çalıştır + çıktıyı oku.** Aşağıdaki upload hücrelerinde parquet kolon adlarına `getattr(r, 'kolon', None)` ile erişiyoruz. Eğer kolon adı (özellikle case) farklıysa burada görülür ve row_builder'ı düzeltme şansın olur.

In [ ]:
PARQUET_META = {}
for name, path in PATHS.items():
    if not os.path.exists(path):
        print(f'✗ skip {name} (missing)')
        continue
    pf = pq.ParquetFile(path)
    cols = pf.schema_arrow.names
    PARQUET_META[name] = {
        'rows': pf.metadata.num_rows,
        'row_groups': pf.metadata.num_row_groups,
        'columns': cols,
    }
    print(f'\n=== {name} ===')
    print(f'  rows: {pf.metadata.num_rows:,}  row_groups: {pf.metadata.num_row_groups}')
    print(f'  cols: {cols}')
    head = next(pf.iter_batches(batch_size=2)).to_pandas()
    with pd.option_context('display.max_columns', None, 'display.width', 200):
        print(f'  head:\n{head}')

print('\n=== ENVANTER beklenen satır sayısı vs gerçek ===')
EXPECTED_ROWS = {
    'fact_method_field_affinity':       390,
    'fact_method_topic_affinity':    65_061,
    'dim_paper_replication':     24_867_210,
    'fact_paper_field':          24_866_945,
    'fact_paper_interdisc':      24_866_945,
    'fact_paper_velocity':       24_867_210,
    'fact_paper_quality_v3':     24_867_210,
    'fact_paper_w_estra':        24_867_210,
    'fact_paper_topic':          69_751_445,
    'fact_paper_metod':          51_785_496,
    'fact_paper_disruption':     24_867_210,
    'fact_paper_beauty':         24_867_210,
    'fact_paper_centrality':    100_982_867,
    'dim_author':                22_649_014,
    'fact_paper_bibcoupling_top50': 643_445_780,
}
for name, exp in EXPECTED_ROWS.items():
    if name not in PARQUET_META:
        print(f'  ✗ {name}: MISSING')
        continue
    got = PARQUET_META[name]['rows']
    delta = got - exp
    flag = '✓' if abs(delta) < max(100, exp*0.001) else '⚠'
    print(f'  {flag} {name:32s} expected={exp:>13,}  got={got:>13,}  Δ={delta:+,}')

## Cell 3 — Migration verify (0005-0008) + 0009 dim_author dinamik üret+apply

**Önkoşul:** Migration 0005-0008 lokalden uygulanmış olmalı (`psql $DB_URL < db/migrations/0005_paper_estra_temporal.sql` vb.). Bu hücre sadece doğrular, uygulamaz.

**0009 dim_author**: 22 kolon parquet'ten okunup CREATE TABLE üretilir (Cell 2 audit'e dayalı). RLS + paper_id REFERENCES YOK (author_id PK), ON DELETE CASCADE author tablosu için anlamsız.

In [ ]:
REQUIRED = ['0005_paper_estra_temporal','0006_paper_metadata','0007_method_centrality','0008_neighbor_bibcoupling']
with psycopg2.connect(DB_URL) as conn, conn.cursor() as cur:
    cur.execute('SELECT version FROM public.schema_migrations;')
    applied = {r[0] for r in cur.fetchall()}
    missing = [m for m in REQUIRED if m not in applied]
    if missing:
        print(f'✗ Missing migrations: {missing}')
        print('  Lokal makinede çalıştır:')
        for m in missing:
            print(f'    psql "$SUPABASE_DB_URL" < db/migrations/{m}.sql')
        raise SystemExit('Migrations not applied')
    print(f'✓ Migrations 0005-0008 hepsi applied')

# === 0009 dim_author: parquet schema → DDL ===
import pyarrow as pa
_PA_TO_PG = {
    pa.string():           'text',
    pa.large_string():     'text',
    pa.bool_():            'boolean',
    pa.int8():             'smallint',
    pa.int16():            'smallint',
    pa.int32():            'integer',
    pa.int64():            'bigint',
    pa.uint8():            'smallint',
    pa.uint16():           'integer',
    pa.uint32():           'bigint',
    pa.uint64():           'bigint',
    pa.float16():          'real',
    pa.float32():          'real',
    pa.float64():          'double precision',
    pa.date32():           'date',
    pa.date64():           'date',
}
def _arrow_to_pg(field):
    t = field.type
    for k, v in _PA_TO_PG.items():
        if t.equals(k):
            return v
    if pa.types.is_timestamp(t): return 'timestamptz'
    if pa.types.is_list(t) or pa.types.is_large_list(t):
        return 'jsonb'  # listeleri JSONB olarak depo
    if pa.types.is_struct(t):
        return 'jsonb'
    return 'text'

pf = pq.ParquetFile(PATHS['dim_author'])
schema = pf.schema_arrow
fields = list(schema)
pk_col = 'author_id'
pk_names = [f.name for f in fields]
assert pk_col in pk_names, f'dim_author parquet schema beklenen `author_id` yok: {pk_names}'

ddl_lines = ['CREATE TABLE IF NOT EXISTS public.dim_author (']
for f in fields:
    pg_type = _arrow_to_pg(f)
    null_cl = ' NOT NULL' if f.name == pk_col else ''
    ddl_lines.append(f'  {f.name} {pg_type}{null_cl},')
ddl_lines.append(f'  PRIMARY KEY ({pk_col})')
ddl_lines.append(');')
ddl = '\n'.join(ddl_lines)

INDEX_SQL = '''
CREATE INDEX IF NOT EXISTS idx_author_h          ON public.dim_author(h_index DESC);
CREATE INDEX IF NOT EXISTS idx_author_top_topic  ON public.dim_author(top_topic);
CREATE INDEX IF NOT EXISTS idx_author_top_method ON public.dim_author(top_method);
ALTER TABLE public.dim_author ENABLE ROW LEVEL SECURITY;
DROP POLICY IF EXISTS author_read_all      ON public.dim_author;
DROP POLICY IF EXISTS author_write_service ON public.dim_author;
CREATE POLICY author_read_all      ON public.dim_author FOR SELECT TO authenticated USING (true);
CREATE POLICY author_write_service ON public.dim_author FOR ALL    TO service_role  USING (true) WITH CHECK (true);
INSERT INTO public.schema_migrations (version, description)
VALUES ('0009_dim_author', 'dim_author (22.65M × 22 dynamic schema from parquet, N15)')
ON CONFLICT (version) DO NOTHING;
'''

print('=== Generated DDL (0009_dim_author) ===')
print(ddl)
print('--- index + RLS ---')
print(INDEX_SQL)

with psycopg2.connect(DB_URL) as conn, conn.cursor() as cur:
    # h_index/top_topic/top_method varsa indexler güvenli; yoksa hata vermesin diye ayrı try.
    cur.execute(ddl)
    try:
        cur.execute(INDEX_SQL)
    except psycopg2.Error as e:
        conn.rollback()
        print(f'⚠ Index/RLS hatası (kolon eksik olabilir): {e}')
        # RLS + migration insert minimal:
        cur.execute("ALTER TABLE public.dim_author ENABLE ROW LEVEL SECURITY;")
        cur.execute("DROP POLICY IF EXISTS author_read_all ON public.dim_author;")
        cur.execute("DROP POLICY IF EXISTS author_write_service ON public.dim_author;")
        cur.execute("CREATE POLICY author_read_all ON public.dim_author FOR SELECT TO authenticated USING (true);")
        cur.execute("CREATE POLICY author_write_service ON public.dim_author FOR ALL TO service_role USING (true) WITH CHECK (true);")
        cur.execute("INSERT INTO public.schema_migrations (version, description) VALUES ('0009_dim_author', 'dim_author (dynamic, partial index)') ON CONFLICT DO NOTHING;")
    conn.commit()
print('✓ migration 0009_dim_author applied')

# DIM_AUTHOR_COLS: row_builder için kullanılacak — parquet kolon sırası
DIM_AUTHOR_COLS = [f.name for f in fields]
print(f'\nDIM_AUTHOR_COLS ({len(DIM_AUTHOR_COLS)}): {DIM_AUTHOR_COLS}')

## Cell 4 — `stream_upload` helper + `_g/_fnum/_iint/_ibool` utils

B-009 pattern (sentence_role notebook'undan birebir taşındı): VALID_PAPER_IDS anti-join + ETA + skipped count + per-batch commit.

**Restart-safe enrichment**: `STATE` (`{DRIVE_ROOT}/_phase3_state/phase3_state.json`) + JSONL log (`phase3_log.jsonl`). Bir tablo `completed` ise upload hücresi skip eder. Crash olursa hücre tekrar çalıştırılır — `ON CONFLICT DO NOTHING` + state[table]=`error` log'unda görünür. State Drive'da kalıcı.

In [ ]:
def _fnum(v):
    if v is None: return None
    try:
        f = float(v)
        return None if math.isnan(f) else f
    except (TypeError, ValueError): return None

def _iint(v):
    if v is None: return None
    try:
        f = float(v)
        return None if math.isnan(f) else int(f)
    except (TypeError, ValueError): return None

def _ibool(v):
    if v is None: return None
    if isinstance(v, bool): return v
    try: return bool(int(v))
    except (TypeError, ValueError): return None

def _txt(v):
    if v is None: return None
    s = str(v)
    return s if s and s != 'nan' else None

# L-021: paper_id/author_id casing defense — bare W123/A123 enforced.
# ENVANTER §181: N09/N09b/N09c parquet'lerde sporadik openalex.org/ prefix gözlendi.
# B-008 sonrası PaperCard bare W standardı; loader her scan_parquet'te normalize eder.
_OA_PREFIXES = ('https://openalex.org/', 'http://openalex.org/', 'openalex.org/')
def _norm_w(v):
    if v is None: return None
    s = str(v).strip()
    if not s or s == 'nan': return None
    for p in _OA_PREFIXES:
        if s.startswith(p):
            s = s[len(p):]; break
    return s if s else None

def _g(r, *names, default=None):
    """row'dan ilk eşleşen kolon adını al — case mismatch tolere eder."""
    for n in names:
        if hasattr(r, n):
            return getattr(r, n)
    return default

def stream_upload(table, path, sql, template, row_builder,
                  pk_filter='paper_id', total_expected=None,
                  batch_rows=BATCH_ROWS, chunk=CHUNK, page_size=PAGE_SIZE,
                  fk_check=True):
    """Restart-safe loader. State checkpoint + JSONL log on Drive.
    pk_filter: 'paper_id' / 'paper_a_b' / 'none'
    """
    global STATE
    tstate = STATE['tables'].get(table, {})
    if tstate.get('status') == 'completed':
        print(f'\n⏭ {table} already completed (rows={tstate.get("rows_done","?"):,}); skip.')
        _log({'phase': 'skip_completed', 'table': table})
        return

    pf = pq.ParquetFile(path)
    total = pf.metadata.num_rows
    print(f'\n=== upload {table}: {total:,} rows ===')
    if total_expected and abs(total - total_expected) > max(100, total_expected * 0.001):
        print(f'  ⚠ row count mismatch envanter: expected ~{total_expected:,}, got {total:,}')

    STATE['tables'][table] = {
        'status': 'in_progress', 'started_at': datetime.now().isoformat(),
        'parquet_rows': total, 'rows_done': 0, 'skipped': 0,
        'pk_filter': pk_filter, 'path': path,
    }
    _state_save(STATE)
    _log({'phase': 'table_start', 'table': table, 'parquet_rows': total, 'pk_filter': pk_filter})

    t0 = time.time()
    rows_done, skipped = 0, 0
    last_state_save = t0
    try:
        with psycopg2.connect(DB_URL) as conn, conn.cursor() as cur:
            for batch in pf.iter_batches(batch_size=batch_rows):
                df = batch.to_pandas()
                built = []
                for r in df.itertuples(index=False):
                    row = row_builder(r)
                    if row is None:
                        skipped += 1; continue
                    if fk_check and pk_filter == 'paper_id':
                        if row[0] not in VALID_PAPER_IDS:
                            skipped += 1; continue
                    elif fk_check and pk_filter == 'paper_a_b':
                        if row[0] not in VALID_PAPER_IDS or row[1] not in VALID_PAPER_IDS:
                            skipped += 1; continue
                    built.append(row)
                for i in range(0, len(built), chunk):
                    execute_values(cur, sql, built[i:i+chunk], template=template, page_size=page_size)
                    rows_done += min(chunk, len(built) - i)
                    elapsed = time.time() - t0
                    rate = rows_done / elapsed if elapsed > 0 else 0
                    eta_s = (total - rows_done) / rate if rate > 0 else 0
                    stamp = datetime.now().strftime('%H:%M:%S')
                    print(f'  [{stamp}] {rows_done:>12,}/{total:,}  '
                          f'({100*rows_done/total:5.1f}%)  rate={rate:>7,.0f}/s  '
                          f'ETA={timedelta(seconds=int(eta_s))}  skip={skipped:,}', flush=True)
                    _log({'phase': 'batch', 'table': table, 'rows_done': rows_done,
                          'skipped': skipped, 'rate_per_s': round(rate, 1),
                          'pct': round(100*rows_done/total, 2)})
                conn.commit()
                # state checkpoint every 30s (avoid Drive write thrash)
                if time.time() - last_state_save > 30:
                    STATE['tables'][table].update({
                        'rows_done': rows_done, 'skipped': skipped,
                        'last_update': datetime.now().isoformat(),
                    })
                    _state_save(STATE)
                    last_state_save = time.time()
                del df, built
    except Exception as e:
        STATE['tables'][table].update({
            'status': 'error', 'error': str(e),
            'rows_done': rows_done, 'skipped': skipped,
            'failed_at': datetime.now().isoformat(),
        })
        _state_save(STATE)
        _log({'phase': 'error', 'table': table, 'error': str(e),
              'rows_done': rows_done, 'skipped': skipped})
        raise

    elapsed_s = int(time.time() - t0)
    print(f'\n✓ {table} uploaded in {timedelta(seconds=elapsed_s)}  (skipped={skipped:,})')
    with psycopg2.connect(DB_URL) as conn, conn.cursor() as cur:
        cur.execute(f'SELECT COUNT(*) FROM public.{table};')
        db_count = cur.fetchone()[0]
    print(f'  DB count: {db_count:,}')

    STATE['tables'][table].update({
        'status': 'completed', 'rows_done': rows_done, 'skipped': skipped,
        'db_count': db_count, 'finished_at': datetime.now().isoformat(),
        'elapsed_s': elapsed_s,
    })
    _state_save(STATE)
    _log({'phase': 'table_done', 'table': table, 'rows_done': rows_done,
          'skipped': skipped, 'db_count': db_count, 'elapsed_s': elapsed_s})


## Cell 5 — Upload **fact_method_field_affinity** (390 satır, <1 dk)
Composite PK (metod_id, primary_field). FK YOK (no FK to paper).

In [ ]:
def _build_mfa(r):
    return (
        _txt(_g(r, 'metod_id')),
        _txt(_g(r, 'primary_field')),
        _iint(_g(r, 'n_papers')) or 0,
        _fnum(_g(r, 'mean_centrality')),
        _fnum(_g(r, 'mean_q_weak')),
        _fnum(_g(r, 'lift')),
    )
MFA_SQL = '''
INSERT INTO public.fact_method_field_affinity
  (metod_id, primary_field, n_papers, mean_centrality, mean_q_weak, lift)
VALUES %s ON CONFLICT (metod_id, primary_field) DO NOTHING;
'''
MFA_TMPL = '(%s,%s,%s,%s,%s,%s)'
stream_upload('fact_method_field_affinity', PATHS['fact_method_field_affinity'],
              MFA_SQL, MFA_TMPL, _build_mfa, pk_filter='none',
              total_expected=390)

## Cell 6 — Upload **fact_method_topic_affinity** (65,061 satır, <1 dk)

In [ ]:
def _build_mta(r):
    return (
        _txt(_g(r, 'metod_id')),
        _txt(_g(r, 'theme_id')),
        _iint(_g(r, 'n_papers')) or 0,
        _fnum(_g(r, 'mean_centrality')),
        _fnum(_g(r, 'mean_q_weak')),
        _fnum(_g(r, 'lift')),
    )
MTA_SQL = '''
INSERT INTO public.fact_method_topic_affinity
  (metod_id, theme_id, n_papers, mean_centrality, mean_q_weak, lift)
VALUES %s ON CONFLICT (metod_id, theme_id) DO NOTHING;
'''
MTA_TMPL = '(%s,%s,%s,%s,%s,%s)'
stream_upload('fact_method_topic_affinity', PATHS['fact_method_topic_affinity'],
              MTA_SQL, MTA_TMPL, _build_mta, pk_filter='none',
              total_expected=65_061)

## Cell 7 — Upload **dim_paper_replication** (24.87M, ~5 dk)
L-022: replication_signal_present **smallint NOT bool** (0/1). 66,368 paper signal=1 (%0.27).

In [ ]:
def _build_replication(r):
    pid = _norm_w(_g(r, 'paper_id'))
    if pid is None: return None
    sig = _g(r, 'replication_signal_present', 'replication_signal', default=0)
    sig_int = _iint(sig)
    if sig_int is None: sig_int = 1 if bool(sig) else 0
    if sig_int not in (0, 1): sig_int = 1 if sig_int else 0
    return (pid, sig_int, _txt(_g(r, 'regex_evidence')))
REPL_SQL = '''
INSERT INTO public.dim_paper_replication (paper_id, replication_signal_present, regex_evidence)
VALUES %s ON CONFLICT (paper_id) DO NOTHING;
'''
REPL_TMPL = '(%s,%s,%s)'
stream_upload('dim_paper_replication', PATHS['dim_paper_replication'],
              REPL_SQL, REPL_TMPL, _build_replication,
              total_expected=24_867_210)

## Cell 8 — Upload **fact_paper_field** (24.87M, ~10 dk)

In [ ]:
def _build_field(r):
    pid = _norm_w(_g(r, 'paper_id'))
    if pid is None: return None
    return (
        pid,
        _txt(_g(r, 'primary_field')),
        _txt(_g(r, 'primary_subfield')),
        _txt(_g(r, 'primary_domain')),
    )
FIELD_SQL = '''
INSERT INTO public.fact_paper_field (paper_id, primary_field, primary_subfield, primary_domain)
VALUES %s ON CONFLICT (paper_id) DO NOTHING;
'''
FIELD_TMPL = '(%s,%s,%s,%s)'
stream_upload('fact_paper_field', PATHS['fact_paper_field'],
              FIELD_SQL, FIELD_TMPL, _build_field,
              total_expected=24_866_945)

## Cell 9 — Upload **fact_paper_interdisc** (24.87M, ~15 dk)

In [ ]:
def _build_interdisc(r):
    pid = _norm_w(_g(r, 'paper_id'))
    if pid is None: return None
    return (
        pid,
        _iint(_g(r, 'n_distinct_domains')) or 0,
        _iint(_g(r, 'n_distinct_fields')) or 0,
        _iint(_g(r, 'n_themes')) or 0,
        _fnum(_g(r, 'rao_stirling')),
    )
INT_SQL = '''
INSERT INTO public.fact_paper_interdisc
  (paper_id, n_distinct_domains, n_distinct_fields, n_themes, rao_stirling)
VALUES %s ON CONFLICT (paper_id) DO NOTHING;
'''
INT_TMPL = '(%s,%s,%s,%s,%s)'
stream_upload('fact_paper_interdisc', PATHS['fact_paper_interdisc'],
              INT_SQL, INT_TMPL, _build_interdisc,
              total_expected=24_866_945)

## Cell 10 — Upload **fact_paper_velocity** (24.87M, ~15 dk)

In [ ]:
def _build_velocity(r):
    pid = _norm_w(_g(r, 'paper_id'))
    if pid is None: return None
    return (
        pid,
        _fnum(_g(r, 'velocity')),
        _fnum(_g(r, 'velocity_pct_in_field_year')),
        _fnum(_g(r, 'age_adjusted_impact')),
        _ibool(_g(r, 'is_too_new')) or False,
    )
VEL_SQL = '''
INSERT INTO public.fact_paper_velocity
  (paper_id, velocity, velocity_pct_in_field_year, age_adjusted_impact, is_too_new)
VALUES %s ON CONFLICT (paper_id) DO NOTHING;
'''
VEL_TMPL = '(%s,%s,%s,%s,%s)'
stream_upload('fact_paper_velocity', PATHS['fact_paper_velocity'],
              VEL_SQL, VEL_TMPL, _build_velocity,
              total_expected=24_867_210)

## Cell 11 — Upload **fact_paper_quality_v3** (24.87M, ~12 dk)

In [ ]:
def _build_quality_v3(r):
    pid = _norm_w(_g(r, 'paper_id'))
    if pid is None: return None
    return (
        pid,
        _fnum(_g(r, 'q_weak')),
        _fnum(_g(r, 'q_weak_low')),
        _fnum(_g(r, 'q_weak_high')),
        _iint(_g(r, 'n_lfs_active')) or 0,
        _fnum(_g(r, 'q_weak_v2')),
    )
Q3_SQL = '''
INSERT INTO public.fact_paper_quality_v3
  (paper_id, q_weak, q_weak_low, q_weak_high, n_lfs_active, q_weak_v2)
VALUES %s ON CONFLICT (paper_id) DO NOTHING;
'''
Q3_TMPL = '(%s,%s,%s,%s,%s,%s)'
stream_upload('fact_paper_quality_v3', PATHS['fact_paper_quality_v3'],
              Q3_SQL, Q3_TMPL, _build_quality_v3,
              total_expected=24_867_210)

## Cell 12 — Upload **fact_paper_w_estra** (24.87M × 15, ~40 dk)
Parquet kolon casing belirsiz — `_g` çoklu alias ile tolere eder. `gate_w_triggered` Plan 1'de hep false.

In [ ]:
def _build_w_estra(r):
    pid = _norm_w(_g(r, 'paper_id'))
    if pid is None: return None
    return (
        pid,
        _fnum(_g(r, 'w_estra', 'w_ESTRA')),
        _fnum(_g(r, 'w_estra_low', 'w_ESTRA_low')),
        _fnum(_g(r, 'w_estra_high', 'w_ESTRA_high')),
        _fnum(_g(r, 'wir', 'wIR', 'w_IR')),
        _fnum(_g(r, 'wir_low', 'wIR_low', 'w_IR_low')),
        _fnum(_g(r, 'wir_high', 'wIR_high', 'w_IR_high')),
        _fnum(_g(r, 'wts', 'wTS', 'w_TS')),
        _fnum(_g(r, 'wts_low', 'wTS_low', 'w_TS_low')),
        _fnum(_g(r, 'wts_high', 'wTS_high', 'w_TS_high')),
        _fnum(_g(r, 'wd', 'wD', 'w_D')),
        _fnum(_g(r, 'wd_low', 'wD_low', 'w_D_low')),
        _fnum(_g(r, 'wd_high', 'wD_high', 'w_D_high')),
        _iint(_g(r, 'n_themes')) or 0,
        _ibool(_g(r, 'gate_w_triggered', 'gate_w')) or False,
    )
W_SQL = '''
INSERT INTO public.fact_paper_w_estra
  (paper_id, w_estra, w_estra_low, w_estra_high,
   wir, wir_low, wir_high, wts, wts_low, wts_high, wd, wd_low, wd_high,
   n_themes, gate_w_triggered)
VALUES %s ON CONFLICT (paper_id) DO NOTHING;
'''
W_TMPL = '(%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)'
stream_upload('fact_paper_w_estra', PATHS['fact_paper_w_estra'],
              W_SQL, W_TMPL, _build_w_estra,
              total_expected=24_867_210)

## Cell 13 — Upload **fact_paper_topic** (~75M rank≤3 loader filter, ~30 dk)
Parquet 69.75M (TÜM ranks); migration CHECK rank IN [1,3] zorunlu, rank>3 satırlar **loader-side filter** ile atılır.
ENVANTER §4: parquet kolon `id` (=paper_id), theme_id, score, rank, is_primary.

In [ ]:
def _build_topic(r):
    pid = _norm_w(_g(r, 'id', 'paper_id'))
    if pid is None: return None
    rank = _iint(_g(r, 'rank'))
    if rank is None or rank < 1 or rank > 3:
        return None  # loader filter rank≤3
    theme = _txt(_g(r, 'theme_id'))
    if theme is None: return None
    return (
        pid, theme,
        _fnum(_g(r, 'score')),
        rank,
        _ibool(_g(r, 'is_primary')) or False,
    )
TOP_SQL = '''
INSERT INTO public.fact_paper_topic (paper_id, theme_id, score, rank, is_primary)
VALUES %s ON CONFLICT (paper_id, theme_id) DO NOTHING;
'''
TOP_TMPL = '(%s,%s,%s,%s,%s)'
stream_upload('fact_paper_topic', PATHS['fact_paper_topic'],
              TOP_SQL, TOP_TMPL, _build_topic,
              total_expected=69_751_445)

## Cell 14 — Upload **fact_paper_metod** (51.79M, ~30 dk)

In [ ]:
def _build_metod(r):
    pid = _norm_w(_g(r, 'paper_id'))
    metod = _txt(_g(r, 'metod_id'))
    if pid is None or metod is None: return None
    return (
        pid, metod,
        _fnum(_g(r, 'score')),
        _ibool(_g(r, 'is_primary')) or False,
        _txt(_g(r, 'confidence_flag')),
        _txt(_g(r, 'source')),
    )
MET_SQL = '''
INSERT INTO public.fact_paper_metod
  (paper_id, metod_id, score, is_primary, confidence_flag, source)
VALUES %s ON CONFLICT (paper_id, metod_id) DO NOTHING;
'''
MET_TMPL = '(%s,%s,%s,%s,%s,%s)'
stream_upload('fact_paper_metod', PATHS['fact_paper_metod'],
              MET_SQL, MET_TMPL, _build_metod,
              total_expected=51_785_496)

## Cell 15 — Upload **fact_paper_disruption** (24.87M, ~20 dk)

In [ ]:
def _build_disruption(r):
    pid = _norm_w(_g(r, 'paper_id'))
    if pid is None: return None
    return (
        pid,
        _iint(_g(r, 'pub_year')),
        _iint(_g(r, 'n_a', 'N_a', 'Na')) or 0,
        _iint(_g(r, 'n_i', 'N_i', 'Ni')) or 0,
        _iint(_g(r, 'n_j', 'N_j', 'Nj')) or 0,
        _iint(_g(r, 'total_window_papers')) or 0,
        _fnum(_g(r, 'cd_5', 'CD_5', 'cd5')),
        _ibool(_g(r, 'cd_undefined', 'cd5_undefined')) or False,
    )
DIS_SQL = '''
INSERT INTO public.fact_paper_disruption
  (paper_id, pub_year, n_a, n_i, n_j, total_window_papers, cd_5, cd_undefined)
VALUES %s ON CONFLICT (paper_id) DO NOTHING;
'''
DIS_TMPL = '(%s,%s,%s,%s,%s,%s,%s,%s)'
stream_upload('fact_paper_disruption', PATHS['fact_paper_disruption'],
              DIS_SQL, DIS_TMPL, _build_disruption,
              total_expected=24_867_210)

## Cell 16 — Upload **fact_paper_beauty** (24.87M, ~25 dk)
year_obs_max = -1 sentinel (atıfsız paper). b_undefined = year_obs_max < 5.

In [ ]:
def _build_beauty(r):
    pid = _norm_w(_g(r, 'paper_id'))
    if pid is None: return None
    return (
        pid,
        _iint(_g(r, 'pub_year')),
        _iint(_g(r, 'c_max')) or 0,
        _iint(_g(r, 't_m')),
        _iint(_g(r, 't_a')),
        _iint(_g(r, 'c_0')) or 0,
        _iint(_g(r, 'total_cites')) or 0,
        _iint(_g(r, 'year_obs_max')),
        _fnum(_g(r, 'b', 'B')),
        _ibool(_g(r, 'b_undefined', 'B_undefined')) or False,
    )
BEAU_SQL = '''
INSERT INTO public.fact_paper_beauty
  (paper_id, pub_year, c_max, t_m, t_a, c_0, total_cites, year_obs_max, b, b_undefined)
VALUES %s ON CONFLICT (paper_id) DO NOTHING;
'''
BEAU_TMPL = '(%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)'
stream_upload('fact_paper_beauty', PATHS['fact_paper_beauty'],
              BEAU_SQL, BEAU_TMPL, _build_beauty,
              total_expected=24_867_210)

## Cell 17 — Upload **fact_paper_centrality** (100M parquet → ~24.87M corpus subset, ~25 dk)
VALID_PAPER_IDS anti-join: parquet 100,982,867 satır; sadece corpus paper_id'lere ait kayıtlar yüklenir (~24.87M). Skipped count ~76M olmalı.

In [ ]:
def _build_centrality(r):
    pid = _norm_w(_g(r, 'paper_id'))
    if pid is None: return None
    return (
        pid,
        _fnum(_g(r, 'pagerank')),
        _iint(_g(r, 'indegree')) or 0,
        _iint(_g(r, 'outdegree')) or 0,
    )
CEN_SQL = '''
INSERT INTO public.fact_paper_centrality (paper_id, pagerank, indegree, outdegree)
VALUES %s ON CONFLICT (paper_id) DO NOTHING;
'''
CEN_TMPL = '(%s,%s,%s,%s)'
stream_upload('fact_paper_centrality', PATHS['fact_paper_centrality'],
              CEN_SQL, CEN_TMPL, _build_centrality,
              total_expected=100_982_867)

## Cell 18 — Upload **dim_author** (22.65M × 22 dynamic, ~90 dk)
Cell 3'te DIM_AUTHOR_COLS parquet schema'sından okundu; row_builder o sırayı izler. FK guard YOK (author_id PK).

In [ ]:
import numpy as np

def _coerce_for_pg(v):
    if v is None: return None
    if isinstance(v, float) and math.isnan(v): return None
    if isinstance(v, (np.floating,)):
        f = float(v); return None if math.isnan(f) else f
    if isinstance(v, (np.integer,)): return int(v)
    if isinstance(v, (np.bool_,)):   return bool(v)
    if isinstance(v, (list, dict, np.ndarray)):
        try: return json.dumps(v.tolist() if hasattr(v, 'tolist') else v)
        except Exception: return json.dumps(str(v))
    return v

def _build_author(r):
    # parquet kolon sırası ile namedtuple — doğrudan eşle
    out = []
    for col in DIM_AUTHOR_COLS:
        v = getattr(r, col, None)
        out.append(_coerce_for_pg(v))
    aid_idx = DIM_AUTHOR_COLS.index('author_id')
    aid = _norm_w(out[aid_idx])
    if aid is None:
        return None
    out[aid_idx] = aid
    return tuple(out)

_cols_csv = ', '.join(DIM_AUTHOR_COLS)
AUTH_SQL = f'INSERT INTO public.dim_author ({_cols_csv}) VALUES %s ON CONFLICT (author_id) DO NOTHING;'
AUTH_TMPL = '(' + ','.join(['%s'] * len(DIM_AUTHOR_COLS)) + ')'
stream_upload('dim_author', PATHS['dim_author'],
              AUTH_SQL, AUTH_TMPL, _build_author, pk_filter='none',
              total_expected=22_649_014)

## Cell 19 — Upload **fact_paper_bibcoupling_top50** (643M, NO FK, ~3 saat)
**EN BÜYÜK.** 5.17 GB parquet, paper-pair tablo (anchor `paper_id` × top-50 `neighbor_id`, `rank` 1..50). NO FK (loader anti-join). VALID_PAPER_IDS hem `paper_id` hem `neighbor_id` için kontrol edilir.

**SCHEMA AUDIT 2026-05-01**: parquet kolonları `paper_id, neighbor_id, raw_count, cosine_score, rank` — ENVANTER §180 iddiasından farklı (manifest > ENVANTER, B42-039).

**Restart:** Crash olursa hücreyi tekrar çalıştır — ON CONFLICT DO NOTHING idempotent.
**Disk:** 4XL + autoscale 80 GB ile rahat. Bu hücre boyunca dashboard'tan disk doluluğu izle.
**Skip rate beklenen**: ~%0.07 (4,713 PaperCard miss × 50 × 2 yön ≈ 470K pair) — ENVANTER §180 doğrulandı.

In [ ]:
def _build_bibcoup(r):
    pid = _norm_w(_g(r, 'paper_id'))
    nid = _norm_w(_g(r, 'neighbor_id'))
    if pid is None or nid is None: return None
    rk = _iint(_g(r, 'rank'))
    if rk is None or rk < 1 or rk > 50: return None
    return (
        pid, nid,
        _iint(_g(r, 'raw_count')) or 0,
        _fnum(_g(r, 'cosine_score')),
        rk,
    )
BIB_SQL = '''
INSERT INTO public.fact_paper_bibcoupling_top50
  (paper_id, neighbor_id, raw_count, cosine_score, rank)
VALUES %s ON CONFLICT (paper_id, neighbor_id) DO NOTHING;
'''
BIB_TMPL = '(%s,%s,%s,%s,%s)'
stream_upload('fact_paper_bibcoupling_top50', PATHS['fact_paper_bibcoupling_top50'],
              BIB_SQL, BIB_TMPL, _build_bibcoup, pk_filter='paper_a_b',
              total_expected=643_445_780,
              batch_rows=250_000)  # büyük dosya: batch küçült (RAM)


## Cell 20 — Verify (KK gate per tablo)

Plan §7: row count ≤%0.05 sapma + FK violation = 0 + INDEX sanity + dağılım plot.

In [ ]:
EXPECTED = {
    'fact_method_field_affinity':       390,
    'fact_method_topic_affinity':    65_061,
    'dim_paper_replication':     24_867_210,
    'fact_paper_field':          24_866_945,
    'fact_paper_interdisc':      24_866_945,
    'fact_paper_velocity':       24_867_210,
    'fact_paper_quality_v3':     24_867_210,
    'fact_paper_w_estra':        24_867_210,
    'fact_paper_topic':         None,  # loader filter rank≤3 → satır sayısı parquet'ten az
    'fact_paper_metod':          51_785_496,
    'fact_paper_disruption':     24_867_210,
    'fact_paper_beauty':         24_867_210,
    'fact_paper_centrality':    None,  # corpus subset ~24.87M
    'dim_author':                22_649_014,
    'fact_paper_bibcoupling_top50': None,  # paper_a+paper_b VALID_PAPER_IDS sonrası
}
with psycopg2.connect(DB_URL) as conn, conn.cursor() as cur:
    print('=== Row counts ===')
    for tbl, exp in EXPECTED.items():
        cur.execute(f'SELECT COUNT(*) FROM public.{tbl};')
        actual = cur.fetchone()[0]
        if exp is None:
            print(f'  • {tbl:32s} actual={actual:>13,}  (expected: anti-join filtered)')
        else:
            delta = actual - exp
            tol = max(100, exp * 0.0005)
            flag = '✓' if abs(delta) < tol else ('⚠' if abs(delta) < exp*0.001 else '✗')
            print(f'  {flag} {tbl:32s} expected={exp:>13,}  actual={actual:>13,}  Δ={delta:+,}')

    print('\n=== Disk usage ===')
    cur.execute('''SELECT relname, pg_size_pretty(pg_total_relation_size(c.oid)) AS size,
                          pg_total_relation_size(c.oid) AS sz_bytes
                   FROM pg_class c JOIN pg_namespace n ON n.oid = c.relnamespace
                   WHERE n.nspname='public' AND c.relkind='r'
                     AND c.relname IN %s ORDER BY sz_bytes DESC;''',
                (tuple(EXPECTED.keys()),))
    for n, sz, _ in cur.fetchall():
        print(f'  {n:32s} {sz}')

    print('\n=== centrality + topic + bibcoupling sanity ===')
    cur.execute('SELECT MIN(pagerank), AVG(pagerank), MAX(pagerank) FROM public.fact_paper_centrality;')
    print(f'  centrality pagerank min/avg/max: {cur.fetchone()}')
    cur.execute('''SELECT rank, COUNT(*) FROM public.fact_paper_topic
                   GROUP BY rank ORDER BY rank;''')
    for rank, c in cur.fetchall():
        print(f'  topic rank={rank}: {c:,}')
    cur.execute('SELECT COUNT(DISTINCT paper_id) FROM public.fact_paper_bibcoupling_top50;')
    print(f'  bibcoup unique paper_id: {cur.fetchone()[0]:,}  (envanter ~15.87M)')

    print('\n=== Migration list ===')
    cur.execute('SELECT version FROM public.schema_migrations ORDER BY applied_at;')
    for (v,) in cur.fetchall():
        print(f'  • {v}')

print('\n✓ Faz 3 verify complete')

# === State + log summary ===
print('\n=== State checkpoint dump ===')
print(_json.dumps(STATE, indent=2, default=str)[:3000])
import subprocess
try:
    n = subprocess.check_output(['wc', '-l', LOG_PATH]).decode().split()[0]
    print(f'\nLog entries: {n} ({LOG_PATH})')
except Exception:
    pass
